# E11 — Official OpenSpecy comparison
Exploratory paired benchmark. Requires review of source units and reference provenance. Do not interpret the previously inspected exposed collection as fresh independent validation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, numpy as np, pandas as pd
ROOT=Path('/content/drive/MyDrive/SeaweedGuard')
REPO=Path('/content/SeaweedGuard')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/peterDataScientia/SeaweedGuard-.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
DATA=ROOT/'02_Preprocessed_Data'
ARCHIVE=Path('/content/drive/MyDrive/Thermodynamics/doi_10_5061_dryad_f1vhhmh59__v20241126.zip')
if not (DATA/'clean_spectra.npz').exists():
    subprocess.run(['python',str(REPO/'scripts/e02_clean.py'),str(ARCHIVE),'--out',str(DATA)],check=True)
d=pd.read_csv(DATA/'clean_index.csv')
z=np.load(DATA/'clean_spectra.npz')
assert np.array_equal(z['hash'],d.hash.to_numpy())
print(d.groupby(['provisional_label','condition']).size())

## Verified intensity units
The original Dryad archive README.md explicitly identifies intensity as percent absorbance. Divide by 100; do not apply transmittance conversion.

In [ ]:
SIGNAL_UNIT='percent_absorbance' # Verified from original Dryad archive README.md
assert SIGNAL_UNIT in ['absorbance','percent_absorbance','percent_transmittance'], 'Confirm original intensity units first'
out=ROOT/'06_OpenSpecy'
qdir=out/'queries';qdir.mkdir(parents=True,exist_ok=True)
test=d.condition.ne('pristine')
assert test.sum()==232
for i in np.flatnonzero(test.to_numpy()):
    a=z['intensity'][i].astype(float)
    if SIGNAL_UNIT=='percent_absorbance': a=a/100
    elif SIGNAL_UNIT=='percent_transmittance':
        assert np.all(a>0), 'Nonpositive transmittance; review spectrum'
        a=-np.log10(a/100)
    pd.DataFrame({'wavenumber':z['wavenumber'],'absorbance':a}).to_csv(qdir/f"{d.hash.iloc[i]}.csv",index=False)
d.loc[test,['hash','provisional_label','condition']].to_csv(out/'truth.csv',index=False)

In [ ]:
import shutil, time
if not shutil.which('Rscript'):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','r-base'],check=True)
subprocess.run(['Rscript','-e','if (!requireNamespace("OpenSpecy", quietly=TRUE)) install.packages("OpenSpecy", repos="https://cloud.r-project.org"); stopifnot(requireNamespace("OpenSpecy", quietly=TRUE))'],check=True)
run_started=time.time()
p=subprocess.run(['Rscript',str(REPO/'openspecy/benchmark.R'),str(ROOT)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
(out/'execution_log.txt').write_text(p.stdout or '')
print(p.stdout or '(No R output)')
fresh_matches=(out/'matches.csv').exists() and (out/'matches.csv').stat().st_mtime>=run_started
if p.returncode:
    print(f'FAILED: R exit code {p.returncode}. See execution_log.txt')
elif fresh_matches:
    print('Matching output generated in this run.')
else:
    print('PAUSED: matching not complete; inspect reference review and R log.')


## Review results
First execution exports reference metadata and deliberately stops for review. Exclude the study collection, ambiguous sources and non-FTIR references. Rerun after marking verified eligible references. Inspect top matches; manually approve a chemistry-level mapping before calculating accuracy.

In [ ]:
if p.returncode==0 and fresh_matches:
    m=pd.read_csv(out/'matches.csv')
    t=pd.read_csv(out/'truth.csv')
    assert set(m.query_id.astype(str))==set(t.hash.astype(str)), 'Query IDs do not match current truth table'
    print(m.head(20))
    print('Match fields:',list(m.columns))
else:
    print('No completed matching output from this run.')
    if (out/'reference_metadata.csv').exists():
        print('Metadata bytes:',(out/'reference_metadata.csv').stat().st_size)
    if (out/'reference_review.csv').exists():
        review=pd.read_csv(out/'reference_review.csv')
        print('Reference candidates:',len(review))
        print('Approved:',review['include'].astype(str).str.lower().eq('true').sum())
print('Final accuracy requires verified label mapping and reference overlap review.')